In [0]:
import time

folder_path = "/Volumes/workspace/karan/ipl/"
file_name = "Ball_By_Ball.csv"

while True:
    files = [f.name for f in dbutils.fs.ls(folder_path)]

    if file_name in files:
        print("File arrived.")
        break

    print("Waiting for file...")
    time.sleep(30)

df = spark.read.option("header", "true").csv(f"{folder_path}/{file_name}")

In [0]:
from pyspark.sql.functions import col, sum, when

source_file = "/Volumes/workspace/karan/ipl/Ball_By_Ball.csv"
target_path = "/Volumes/workspace/karan/ipl/Ball_By_Ball_tgt"

# Read source
df = spark.read.option("header", "true").csv(source_file)

# Transformation
df2 = (
    df.filter(col("Over_id") == 1)
      .groupBy("Match_id", "Innings_No")
      .agg(
          sum(
              when(col("Runs_Scored") == 6, 1)
              .otherwise(0)
          ).alias("sixes")
      )
      .orderBy(col("sixes").desc())
)

display(df2)

# Append CSV data
(
    df2.write
       .mode("append")
       .option("header", "true")
       .csv(target_path)
)

print("✅ CSV data appended")

# Append into UC Table
(
    df2.write
       .mode("append")
       .saveAsTable("karan.ball_by_ball_tgt")
)

print("✅ Data appended to table karan.ball_by_ball_tgt")

# Verify CSV output
csv_files = dbutils.fs.ls(target_path)

part_files = [f.name for f in csv_files if "part-" in f.name]

if len(part_files) > 0:
    print(f"✅ Output verified. Found {len(part_files)} CSV part files.")
else:
    raise Exception("❌ No output file found.")

# Verify table load
table_count = spark.table("karan.ball_by_ball_tgt").count()

print(f"✅ Table verification successful. Total rows: {table_count}")

# Delete source file
dbutils.fs.rm(source_file)

# Verify deletion
folder_files = [f.name for f in dbutils.fs.ls("/Volumes/workspace/karan/ipl/")]

if "Ball_By_Ball.csv" not in folder_files:
    print("✅ Source file deleted successfully.")
else:
    raise Exception("❌ Source file still exists.")

print("🎉 Pipeline completed successfully")

In [0]:
# ============================================================
# Daily SCD Type 2 load  ->  workspace.default.sctype2
# Tracks changes in ANY of: name, email, city  (full history kept)
# Schedule this notebook as a daily Databricks Workflow job
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import date

# ---------- CONFIG ----------
SRC       = "/Volumes/workspace/default/karan/scdtyp2/customer_initial.csv"
TABLE     = "workspace.default.sctype2"      # catalog.schema.table
HIGH_DATE = "9999-12-31"                     # end_date meaning "still current"

# Columns whose change must create a NEW history row.
# Add or remove a column here and the whole job adapts (one place to edit).
TRACKED = ["name", "email", "city"]

# Effective date = the day this file is processed.
# To backfill a past day, replace with a string like "2025-06-01".
EFFECTIVE_DATE = date.today().isoformat()

# Builds: NOT (a.name <=> b.name) OR NOT (a.email <=> b.email) OR NOT (a.city <=> b.city)
# <=> is NULL-safe equality, so value->NULL and NULL->value are detected too.
def diff_cond(left, right):
    return " OR ".join([f"NOT ({left}.{c} <=> {right}.{c})" for c in TRACKED])

# ---------- STEP 1: create target table if it does not exist ----------
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        customer_id INT,
        name        STRING,
        email       STRING,
        city        STRING,
        start_date  DATE,       -- day this version became valid
        end_date    DATE,       -- last day this version was valid (9999-12-31 = open)
        is_current  BOOLEAN
    ) USING DELTA
""")

# ---------- STEP 2: read today's file ----------
schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("name",        StringType()),
    StructField("email",       StringType()),
    StructField("city",        StringType()),
])

src = (spark.read
       .option("header", True)
       .schema(schema)                                   # explicit schema: faster and safer
       .csv(SRC)
       # trim spaces so "Pune " vs "Pune" is not treated as a change
       .select("customer_id",
               *[F.trim(F.col(c)).alias(c) for c in TRACKED])
       .withColumn("effective_date", F.lit(EFFECTIVE_DATE).cast("date")))

# ---------- STEP 3: data quality checks ----------
# 3a. MERGE fails if one customer_id appears twice in the source
dups = src.groupBy("customer_id").count().filter("count > 1")
if dups.count() > 0:
    dups.show()
    raise ValueError("Duplicate customer_id in source file. Fix the file and rerun.")

# 3b. customer_id must not be NULL (a NULL key can never match)
if src.filter("customer_id IS NULL").count() > 0:
    raise ValueError("NULL customer_id found in source file.")

# ---------- STEP 4: find customers where ANY tracked column changed ----------
current = spark.table(TABLE).filter("is_current = true")

changed = (src.alias("u")
           .join(current.alias("t"), "customer_id")      # existing customers only
           .where(diff_cond("u", "t"))                   # name OR email OR city differs
           .select("customer_id", "u.name", "u.email", "u.city", "effective_date"))

# ---------- STEP 5: build staged rows ----------
# Copy A: every source row, merge_key = customer_id  -> can match and close the old row
# Copy B: changed rows only, merge_key = NULL        -> never matches, so it inserts the new version
staged = (src.withColumn("merge_key", F.col("customer_id"))
          .unionByName(changed.withColumn("merge_key", F.lit(None).cast("int"))))

# ---------- STEP 6: one atomic MERGE ----------
(DeltaTable.forName(spark, TABLE).alias("t")
    .merge(staged.alias("s"),
           "t.customer_id = s.merge_key AND t.is_current = true")
    # any tracked column changed -> close current row the day before the new one starts
    .whenMatchedUpdate(
        condition=diff_cond("t", "s"),
        set={"end_date":   "date_sub(s.effective_date, 1)",
             "is_current": "false"})
    # new customer OR new version of an existing customer -> insert
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "name":        "s.name",
        "email":       "s.email",
        "city":        "s.city",
        "start_date":  "s.effective_date",
        "end_date":    f"CAST('{HIGH_DATE}' AS DATE)",
        "is_current":  "true"})
    .execute())

# ---------- STEP 7: integrity checks (job fails loudly if history is broken) ----------
# 7a. Never more than ONE current row per customer
bad_current = spark.sql(f"""
    SELECT customer_id, COUNT(*) AS c FROM {TABLE}
    WHERE is_current = true GROUP BY customer_id HAVING COUNT(*) > 1
""")
if bad_current.count() > 0:
    bad_current.show()
    raise ValueError("Integrity error: more than one current row for a customer.")

# 7b. No overlapping date ranges for the same customer
overlaps = spark.sql(f"""
    SELECT a.customer_id FROM {TABLE} a JOIN {TABLE} b
      ON a.customer_id = b.customer_id
     AND a.start_date < b.start_date
     AND a.end_date  >= b.start_date
""")
if overlaps.count() > 0:
    overlaps.show()
    raise ValueError("Integrity error: overlapping start/end date ranges.")

# ---------- STEP 8: audit - what did today's run do? ----------
print(f"=== Run date: {EFFECTIVE_DATE} ===")

print("Records INSERTED today (new customers and new versions):")
spark.sql(f"""
    SELECT * FROM {TABLE}
    WHERE start_date = DATE'{EFFECTIVE_DATE}'
    ORDER BY customer_id
""").show(truncate=False)

print("Records CLOSED today (old versions expired):")
spark.sql(f"""
    SELECT * FROM {TABLE}
    WHERE end_date = date_sub(DATE'{EFFECTIVE_DATE}', 1)
      AND is_current = false
    ORDER BY customer_id
""").show(truncate=False)

# Delta's own commit metrics for the MERGE
(DeltaTable.forName(spark, TABLE).history(1)
    .select("version", "timestamp", "operation",
            F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
            F.col("operationMetrics.numTargetRowsUpdated").alias("closed"))
    .show(truncate=False))

# ---------- STEP 9: handy queries ----------
# Current snapshot
spark.sql(f"SELECT * FROM {TABLE} WHERE is_current = true ORDER BY customer_id").show()

# Full history of one customer
spark.sql(f"SELECT * FROM {TABLE} WHERE customer_id = 1 ORDER BY start_date").show()

# Point in time: what did each customer look like on a given date?
spark.sql(f"""
    SELECT customer_id, name, email, city FROM {TABLE}
    WHERE DATE'2026-01-01' BETWEEN start_date AND end_date
    ORDER BY customer_id
""").show()

In [0]:
# ============================================================
# SCD Type 2 with TIMESTAMPS  ->  workspace.default.sctype2_ts
# Handles ANY number of changes per customer on the SAME day,
# in ANY tracked column (name, email, city). Full history kept.
#
# Source file must carry the time each change happened (change_ts).
# One set-based pass + ONE atomic MERGE (no loops).
#
# Interval rule (half-open):  start_ts <= t < end_ts
#   closed row's end_ts = start_ts of the NEXT version
#   current row's end_ts = 9999-12-31 00:00:00
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ---------- CONFIG ----------
SRC     = "/Volumes/workspace/default/karan/scdtyp2/customer_changes_day1.csv"   # change per run
TABLE   = "workspace.default.sctype2_ts"      # new table (timestamp columns)
HIGH_TS = "9999-12-31 00:00:00"               # end_ts meaning "still current"

# Columns whose change creates a NEW history row (edit this list only)
TRACKED = ["name", "email", "city"]

# ---------- STEP 1: create target table if missing ----------
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        customer_id INT,
        name        STRING,
        email       STRING,
        city        STRING,
        start_ts    TIMESTAMP,    -- moment this version became valid (inclusive)
        end_ts      TIMESTAMP,    -- moment the next version took over (exclusive)
        is_current  BOOLEAN
    ) USING DELTA
""")

# ---------- STEP 2: read the change file (many rows per customer allowed) ----------
schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("name",        StringType()),
    StructField("email",       StringType()),
    StructField("city",        StringType()),
    StructField("change_ts",   StringType()),     # 'yyyy-MM-dd HH:mm:ss'
])

src = (spark.read.option("header", True).schema(schema).csv(SRC)
       .select("customer_id",
               *[F.trim(F.col(c)).alias(c) for c in TRACKED],   # trim: "Pune " != "Pune" otherwise
               F.to_timestamp("change_ts").alias("ts")))

# ---------- STEP 3: data quality checks ----------
if src.filter("customer_id IS NULL OR ts IS NULL").count() > 0:
    raise ValueError("NULL customer_id or unparseable change_ts in source file.")

# Two rows with the same customer + same timestamp: order is unknowable
dups = src.groupBy("customer_id", "ts").count().filter("count > 1")
if dups.count() > 0:
    dups.show()
    raise ValueError("Same customer_id + change_ts appears twice. Fix the file.")

current = spark.table(TABLE).filter("is_current = true")

# Every change must be NEWER than the customer's current version.
# Fails on: file already loaded (rerun) OR late-arriving older data.
late = (src.alias("s").join(current.alias("t"), "customer_id")
        .filter("s.ts <= t.start_ts")
        .select("customer_id", "s.ts", "t.start_ts"))
if late.count() > 0:
    late.show()
    raise ValueError("Change older than current version: file already loaded, or late data.")

# ---------- STEP 4: drop no-op rows (nothing really changed) ----------
# Trick: put the customer's CURRENT table row in the sequence as a "seed",
# then compare every row with the row just before it (lag) in time order.
seed = (current.join(src.select("customer_id").distinct(), "customer_id", "left_semi")
        .select("customer_id", *TRACKED,
                F.col("start_ts").alias("ts"),
                F.lit(True).alias("is_seed")))

rows = src.select("customer_id", *TRACKED, "ts", F.lit(False).alias("is_seed"))

w = Window.partitionBy("customer_id").orderBy("ts")       # shuffle by customer_id

flagged = seed.unionByName(rows).withColumn("rn", F.row_number().over(w))
for c in TRACKED:
    flagged = flagged.withColumn(f"prev_{c}", F.lag(c).over(w))

# NULL-safe "any tracked column differs from the previous row"
diff_any = F.lit(False)
for c in TRACKED:
    diff_any = diff_any | ~F.col(c).eqNullSafe(F.col(f"prev_{c}"))

# keep: real rows only, and (first row of a brand-new customer OR something changed)
kept = flagged.filter((~F.col("is_seed")) & ((F.col("rn") == 1) | diff_any))

# ---------- STEP 5: give every kept row its start_ts / end_ts ----------
# end_ts of a version = start of the next version (lead). Last one stays open.
versions = (kept.select("customer_id", *TRACKED, "ts")
            .withColumn("next_ts", F.lead("ts").over(w))
            .withColumn("end_ts", F.coalesce("next_ts", F.lit(HIGH_TS).cast("timestamp")))
            .withColumn("is_current", F.col("next_ts").isNull()))

# ---------- STEP 6: staged rows (close-old + insert-new in ONE merge) ----------
# Copy A: one row per affected customer, merge_key = id -> matches current row -> CLOSE it
#         at the moment of the customer's FIRST kept change in this file.
# Copy B: every new version, merge_key = NULL -> never matches -> INSERT.
copy_close = (versions.groupBy("customer_id").agg(F.min("ts").alias("close_ts"))
              .select(F.col("customer_id").alias("merge_key"),
                      "customer_id",
                      *[F.lit(None).cast("string").alias(c) for c in TRACKED],
                      F.lit(None).cast("timestamp").alias("start_ts"),
                      F.lit(None).cast("timestamp").alias("end_ts"),
                      F.lit(None).cast("boolean").alias("is_current"),
                      "close_ts"))

copy_insert = versions.select(F.lit(None).cast("int").alias("merge_key"),
                              "customer_id", *TRACKED,
                              F.col("ts").alias("start_ts"),
                              "end_ts", "is_current",
                              F.lit(None).cast("timestamp").alias("close_ts"))

staged = copy_close.unionByName(copy_insert)

# ---------- STEP 7: one atomic MERGE ----------
insert_values = {"customer_id": "s.customer_id",
                 **{c: f"s.{c}" for c in TRACKED},
                 "start_ts": "s.start_ts",
                 "end_ts": "s.end_ts",
                 "is_current": "s.is_current"}

(DeltaTable.forName(spark, TABLE).alias("t")
    .merge(staged.alias("s"), "t.customer_id = s.merge_key AND t.is_current = true")
    # old current row: close it when the customer's first new version starts
    .whenMatchedUpdate(set={"end_ts": "s.close_ts", "is_current": "false"})
    # Copy B rows only. (Copy A of a brand-new customer matches nothing; the
    # condition stops it from being inserted as an empty row.)
    .whenNotMatchedInsert(condition="s.merge_key IS NULL", values=insert_values)
    .execute())

# ---------- STEP 8: integrity checks (job fails loudly if history is broken) ----------
# 8a. never more than ONE current row per customer
bad_current = spark.sql(f"""
    SELECT customer_id, COUNT(*) AS c FROM {TABLE}
    WHERE is_current = true GROUP BY customer_id HAVING COUNT(*) > 1""")
if bad_current.count() > 0:
    bad_current.show()
    raise ValueError("Integrity error: more than one current row for a customer.")

# 8b. no gaps and no overlaps: each row's end_ts must equal the next row's start_ts
chain = (spark.table(TABLE)
         .withColumn("next_start",
                     F.lead("start_ts").over(Window.partitionBy("customer_id").orderBy("start_ts"))))
broken = chain.filter(F.col("next_start").isNotNull() & (F.col("end_ts") != F.col("next_start")))
if broken.count() > 0:
    broken.show(truncate=False)
    raise ValueError("Integrity error: gap or overlap in history.")

# 8c. no zero-length or negative-length versions
if spark.table(TABLE).filter("start_ts >= end_ts").count() > 0:
    raise ValueError("Integrity error: a version has start_ts >= end_ts.")

# ---------- STEP 9: audit ----------
print("Delta commit metrics for this run:")
(DeltaTable.forName(spark, TABLE).history(1)
    .select("version", "timestamp", "operation",
            F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
            F.col("operationMetrics.numTargetRowsUpdated").alias("closed"))
    .show(truncate=False))

print("Full history of customers present in today's file:")
(spark.table(TABLE)
    .join(src.select("customer_id").distinct(), "customer_id", "left_semi")
    .orderBy("customer_id", "start_ts")
    .show(truncate=False))

# ---------- STEP 10: handy queries ----------
# Current snapshot
spark.sql(f"SELECT * FROM {TABLE} WHERE is_current = true ORDER BY customer_id").show()

# Point in time (note: >= start AND < end, because intervals are half-open)
spark.sql(f"""
    SELECT customer_id, name, email, city FROM {TABLE}
    WHERE TIMESTAMP'2026-10-26 13:00:00' >= start_ts
      AND TIMESTAMP'2026-10-26 13:00:00' <  end_ts
    ORDER BY customer_id
""").show()